# Homework 3 #

This homework focusses on problems using matrices and solving systems of equations.

In [ ]:
# reload_ext (instead of load_ext) to supress warning
# autoreload 2 requests all modules be reloaded every time
%reload_ext autoreload 
%autoreload 2 

# load homework3
import homework3 as hw

## Problem 1

Write a function named `problem1()` which takes no arguments and does the following:
* Create your own 4x4 2D array with elements 0,1,...15; swap the rows and columns (take the tranpose); print the result
* Generate another 4 (row)x5 (column) 2-D array with elements 0,1,...19; remove the 5th column to make it a 4x4 array; print the result
* Multiply these two 4x4 arrays together (elementwise) and print the result
* Matrix multiply these two 4x4 arrays together and print the result

I recommend using NumPy functions, which should make this easy.

In [ ]:
# Confirm it works:
hw.problem1()
# You should obtain:
# [[ 0  4  8 12]
#  [ 1  5  9 13]
#  [ 2  6 10 14]
#  [ 3  7 11 15]]
# [[ 0  1  2  3]
#  [ 5  6  7  8]
#  [10 11 12 13]
#  [15 16 17 18]]
# [[  0   4  16  36]
#  [  5  30  63 104]
#  [ 20  66 120 182]
#  [ 45 112 187 270]]
# [[280 304 328 352]
#  [310 338 366 394]
#  [340 372 404 436]
#  [370 406 442 478]]

### Radiation Transfer Review

In astrophysics, we often want encounter problems where we need to solve for the steady-state structure of an object, like a star or calculated the run of a physical variable as a function of position.  Let's consider the example of radiation transfer through an atmosphere.  The steady state radiation transfer equation for the specific intensity $I_\nu$ in the presence of isotropic scattering is
\begin{equation}
\mu\frac{\partial I_\nu}{\partial z} = \eta_\nu - (\alpha_\nu +\sigma_\nu) I_\nu + \sigma_\nu J_\nu,
\end{equation}
where $\mu$ is the cosine of the angle the ray makes with the vertical, $z$ is the height in the atmosphere, $\eta_\nu$ is the emissivity, $\alpha_\nu$ and $\sigma_\nu$ the extinction coefficients associated with absorption and scattering, and $J_\nu =(4\pi)^{-1} \int I_\nu d\Omega$ is the angle averaged intensity at frequency $\nu$.

Define variables $\epsilon_\nu = \sigma_\nu/(\sigma_\nu+\alpha_\nu)$, and the optical depth $d \tau_\nu = -(\alpha_\nu+\sigma_\nu) dz$. Also, use Kirchoff's law to relate $\eta_\nu = B_\nu \alpha_\nu$, where
\begin{equation}
B_\nu=\frac{2 h}{c^2}\frac{\nu^3}{\exp(h\nu/k_B T) -1}.
\end{equation}
Then the transfer equation becomes
\begin{equation}
\mu\frac{\partial I_\nu}{\partial \tau_\nu} = I_\nu - \epsilon_\nu B_\nu -(1-\epsilon_\nu)J_\nu.
\end{equation}

In general, the transfer equation is solve (analytically) for all rays at different angles to the vertical or some number of discrete values. In the two stream approximation, the transfer equation is solved for one upward and one downward going ray corresponding to $\mu = \pm 1/\sqrt{3}$ so that $I^+_\nu = I_\nu(\mu=+1/\sqrt{3})$ and $I^-_\nu = I_\nu(\mu=-1/\sqrt{3})$. Hence, the transfer equation becomes
\begin{equation}
\pm\frac{1}{\sqrt{3}}\frac{\partial I^\pm_\nu}{\partial \tau_\nu} = I^\pm_\nu - \epsilon_\nu B_\nu -(1-\epsilon_\nu)J_\nu.
\end{equation}


Approximating $J_\nu = (I^+_\nu + I^-_\nu)/2$ and $H_\nu = (I^+_\nu - I^-_\nu)/(2\sqrt{3})$, we can add or subtract the equations for $I^\pm_\nu$ to obtain
\begin{equation}
\frac{\partial H_\nu}{\partial \tau_\nu} = \epsilon_\nu (J_\nu - B_\nu),
\end{equation}
and
\begin{equation}
\frac{\partial J_\nu}{\partial \tau_\nu} = 3 H_\nu.
\end{equation}
Taking the derivative of the second equation with respect to $\tau_\nu$ and inserting the first equation yields
\begin{equation}
\frac{\partial^2 J_\nu}{\partial \tau_\nu^2} = 3\epsilon_\nu (J_\nu - B_\nu). \qquad (1)
\end{equation}

### Boundary Conditions

At the top of the atmosphere, we assume no incoming radiation so that $I^-_\nu =0$. This means that
\begin{equation}
J_\nu = \sqrt{3} H_\nu = \frac{1}{\sqrt{3}} \frac{\partial J_\nu}{\partial \tau_\nu}.
\end{equation}

At the lower boundary, we assume we have $J_\nu = B_\nu$.  We now have everything we need to solve the transfer equation for $J_\nu$ assuming we know $B_\nu$ as a function of $\tau_\nu$.

## Problem 2

Write functions for computing $B_\nu$, $\alpha_\nu$, and $\sigma_\nu$ given density $\rho(z)$ and temperature $T(z)$.  Assume that $\alpha_\nu = \kappa^{ff}_\nu \rho$ and $\sigma_\nu = \kappa_{\rm es} \rho$.  For simplicity, assume the atmosphere is completely ionized H so that
\begin{equation}
\sigma_\nu = \frac{\sigma_{\rm T}}{m_p} \rho
\end{equation}
and approximate free-free absorption by
\begin{equation}
\alpha_\nu =\frac{4 e^6}{3 m_p^2 m_e h c} \left(\frac{2\pi}{3 k_B m_e} \right)^{1/2} \rho^2 T^{-1/2} \nu^{-3} \left[1- \exp(- h\nu/k_B T)\right]
\end{equation}
which sets the Gaunt factor to 1. Here $e$ is the electron charge, $m_p$ is the proton mass, $k_B$ is Boltzmann's constant, $h$ is Planck's constant, and $c$ is the speed of light. I recommend using the `astropy.constants` module for setting the physical constants: https://docs.astropy.org/en/stable/constants/index.html

These functions are:
* `absorption_extinction()`, which takes arguments $\rho$, $T$, and $\nu$
* `scattering_extinction()`, which takes arguments $\rho$
* `planck()`, which takes arguments $T$ and $\nu$

The arguments $\rho$ and $T$ will be NumPy arrays.  $\nu$ will be a float (a single specific frequency) in the matrix equation we derive below.  But for testing purposes, we write the equation so that $\rho$ and $T$ can be floats with $\nu$ is an array.  You shouldn't have to do anything special to accomplish this -- NumPy's array class make it very easy for us to write expressions that can take either floats or arrays that and perform elementwise arithematic.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
nu = np.logspace(14.,17.,100)
temp = 3.e5
bnu = hw.planck(temp, nu)
plt.plot(nu,bnu)
plt.xscale('log')
plt.yscale('log');

Your result should look like:

![figure1](hw3fig1.png)

In [ ]:
%matplotlib inline
nu = np.logspace(14.,17.,100)
rho = 1.e-9
temp = 3.e5
anu = hw.absorption_extinction(temp, rho, nu)
print(hw.scattering_extinction(rho)) # 3.977263862130838e-10
plt.plot(nu,anu)
plt.xscale('log')
plt.yscale('log');

Your result should look like:

![figure1](hw3fig2.png)

## Problem 3

We want to solve the transfer equation to compute $J_\nu^{(i)}$ on a non-uniform grid of $\tau_\nu$ running from $\tau_\nu^{(0)}$ to $\tau_\nu^{(n-1)}$. Take the first step towards this by creating a matrix equation of the form
\begin{equation}
{\sf A} \mathbf{J} = \mathbf{b},
\end{equation}
where the vector $\mathbf{J}$ is the value a $J_\nu^{(i)}$ at the $n$ points.

#### Discretization

One way of doing this is to rewrite equation (1) as:
\begin{equation}
-\frac{1}{3 \epsilon_\nu^{(i)}}\left(\frac{\partial^2 J_\nu}{\partial \tau_\nu^2}\right)^{(i)} + J_\nu^{(i)} = B_\nu^{(i)}.
\end{equation}

Since we are (possibly) dealing with a non-uniform grid, we need to be careful with our discretization of the second derivative. Lets try the following
\begin{equation}
\left(\frac{\partial J_\nu}{\partial \tau_\nu}\right)^{(i+1/2)} \rightarrow \frac{J_\nu^{(i+1)}-J_\nu^{(i)}}{\tau_\nu^{(i+1)}-\tau_\nu^{(i)}}.
\end{equation}
Here, the index notation means that integers $(i)$ are evaluated on the grid of points $z$ and 1/2 integers $(i \pm 1/2)$ are evaluated at midpoints.  We can then approximate
\begin{equation}
\left(\frac{\partial^2 J_\nu}{\partial \tau_\nu^2}\right)^{(i)} \rightarrow \frac{\left(\frac{\partial J_\nu}{\partial \tau_\nu}\right)^{(i+1/2)} - \left(\frac{\partial J_\nu}{\partial \tau_\nu}\right)^{(i-1/2)} }{\tau_\nu^{(i+1/2)}-\tau_\nu^{(i-1/2)}}. \qquad (2)
\end{equation}
To estimate variables at the half-integer points, we can take a simple average e.g. $\tau_\nu^{(i+1/2)} = (\tau_\nu^{(i+1)}+\tau_\nu^{(i)})/2$ and $\tau_\nu^{(i-1/2)} = (\tau_\nu^{(i)}+\tau_\nu^{(i-1)})/2$.

For the boundary condition at $i=0$, we can approximate the derivative as a forward difference
\begin{equation}
J_\nu^{(0)}=\frac{1}{\sqrt{3}}\frac{J_\nu^{(1)}-J_\nu^{(0)}}{\tau_\nu^{(1)}-\tau_\nu^{(0)}}, \quad (3)
\end{equation}
and for $i=n-1$ we simply have 
\begin{equation}
J_\nu^{(n-1)}=B_\nu^{(n-1)}. \quad (4)
\end{equation}

Write a function to return the matrix ${\sf A}$ and vector $\mathbf{b}$. This requires using eqs. (2) to work out expresions for the matrix elements multiplying $J_\nu^{(i+1)}$, $J_\nu^{(i)}$, and $J_\nu^{(i-1)}$ for each $i = 1, ..., n-2$ and eqs. (3) and (4) to work out $i=0,n-1$.  The vector $\mathbf{b}$ is just $B_\nu^{(i)}$ for all $i$ except $i=0$, where it is 0.

Create a function `transfer_matrices()` which takes the following arguments:
* `tau`: an $n$ element array of optical depths $\tau_\nu$
* `bnu`: an $n$ element array of the Planck function $B_\nu$
* `enu`: an $n$ element array of the opacity fraction $\epsilon_\nu$

and returns $n\times n$ matrix ${\sf A}$ and $n$ element vector $\mathbf{b}$

In [ ]:
# set atmosphere parameters
zmax = 1.e10
zmin = 0.
taumin = 1.e-3
taumax = 1.e3
kapes = 0.33
temp0 = 1.e5

# Create arrays
n = 200
z = np.linspace(zmin, zmax ,n)
l0 = (zmax - zmin) / np.log(taumax/taumin)
rho = taumin / (l0 * kapes) * np.exp((z-zmin)/l0)
temp = np.ones(n)*temp0

# evaluate and store functions of opacities, etc.
nu = 1.e15
abnu = hw.absorption_extinction(temp, rho, nu)
scnu = hw.scattering_extinction(rho)
bnu = hw.planck(temp, nu)
enu = abnu / (scnu + abnu)
    
# compute optical depth
dz = np.gradient(z)
tau = np.empty(n)
tau[0] = taumin
for i in range(1,n):
    tau[i] = tau[i-1] + 0.5*(abnu[i-1]+scnu[i-1]+abnu[i]+scnu[i])*dz[i]

In [ ]:
# compute matrices
amat, bvec = hw.transfer_matrices(tau, bnu, enu)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
# solve equation and plot the result
plt.xlabel(r'$\tau_\nu$')
plt.ylabel(r'$J_\nu/B_\nu,~H_\nu/B_\nu$')

jnu = np.linalg.solve(amat, bvec)
hnu = np.gradient(jnu)/np.gradient(tau)/3
bnu = hw.planck(temp, nu)

plt.plot(tau, jnu/bnu)
plt.plot(tau, hnu/bnu)
plt.xscale('log');

The flux should be low in the deep interior and the mean intensity should approach the Planck function.  Near the surface, the mean intensity should fall and the flux should rise to be comparable to (but not exceed) the mean intensity. Is this what you find? Can we check the result more quantitatively?

Yes! If $\epsilon_\nu$ and $B_\nu$ are both constants, we can solve eq. (1) analytically.  We find
\begin{equation}
J_\nu = B_\nu \left[1 -\frac{\exp\left(-\sqrt{3 \epsilon_\nu} \tau_\nu\right)}{1+\sqrt{\epsilon_\nu}} \right]
\end{equation}

In [ ]:
# compute matrices
eps = np.ones(n)*0.5
amat, bvec = hw.transfer_matrices(tau, bnu, eps)
jnu = np.linalg.solve(amat, bvec)
jnu_an = 1-np.exp(-np.sqrt(3*eps)*tau)/(1+np.sqrt(eps))
plt.plot(tau, jnu/bnu)
plt.plot(tau, jnu_an)
plt.xscale('log');

The results should agree very well.

## Problem 4

Now that we have the solution for $J_\nu$ we can actually solve the transfer equation directly. Write a function to solve the radiation transfer equation
\begin{equation}
\mu\frac{\partial I_\nu}{\partial \tau_\nu} = I_\nu - \epsilon_\nu B_\nu -(1-\epsilon_\nu)J_\nu.
\end{equation}

One usually defines the source function
\begin{equation}
S_\nu = \epsilon_\nu B_\nu + (1-\epsilon_\nu) J_\nu
\end{equation}

Using an integrating factor $\exp(-\tau_\nu/\mu)$ and this definitions of $S_\nu$, we can derive an analytic solution to the transfer equation! This is called the formal solution:
\begin{equation}
I_\nu(\tau_\nu) = I_\nu(\tau_{\rm r})\exp\left[-(\tau_{\rm r} - \tau_\nu)/\mu\right]+\int^{\tau_{\rm r}}_{\tau_\nu} \frac{S_\nu}{\mu}\exp\left[-(\tau_\nu' - \tau_\nu)/\mu\right] d\tau_\nu'
\end{equation}
Here $\tau_{\nu}$ is the value where one wants to know $I_\nu$ and $\tau_{\rm r}$ is a reference value where $I_\nu$ is already known. Often, the reference values is taken to be some large optical depth deep in the atmosphere where $I_\nu \simeq B_\nu$.

Write a function `formal_solution()` that takes as arguments
* `mu`: the cosine of the angle for which the transfer is viwed relative to vertical
* `tau`: the optical depth grid used for computing the transfer equation
* `snu`: the source function $S_\nu$ evaluated on the same `tau` grid
* `ibase`: the intensity at the base of the atmosphere

and returns the intensity corresponding to the surface at `tau[0]`

The integral in this equation can be solved using quadrature on a tabulated grid.  I recommend using the `scipy.integrate.simpson()` function.

In [ ]:
# (re)set atmosphere parameters, if needed
import numpy as np
zmax = 1.e10
zmin = 0.
taumin = 1.e-3
taumax = 1.e3
kapes = 0.33
temp0 = 1.e5

# Create arrays
n = 200
z = np.linspace(zmin, zmax ,n)
l0 = (zmax - zmin) / np.log(taumax/taumin)
rho = taumin / (l0 * kapes) * np.exp((z-zmin)/l0)
temp = np.ones(n)*temp0

# evaluate and store functions of opacities, etc.
nu = 1.e15
abnu = hw.absorption_extinction(temp, rho, nu)
scnu = hw.scattering_extinction(rho)
bnu = hw.planck(temp, nu)

# compute optical depth
dz = np.gradient(z)
tau = np.empty(n)
tau[0] = taumin
for i in range(1,n):
    tau[i] = tau[i-1] + 0.5*(abnu[i-1]+scnu[i-1]+abnu[i]+scnu[i])*dz[i]

In [ ]:
# test it with a blackbody for snu

# the result should be close to 1 regardless of mu
inu = hw.formal_solution(0.3, tau, bnu, bnu[-1])
print(inu/bnu[0])
inu = hw.formal_solution(0.8, tau, bnu, bnu[-1])
print(inu/bnu[0])

Our ultimate interest is to get a solution of the spectrum of emission as a function of frequency for a larger number of frequency points. We obviously want our solve to be as fast as possible.  In this case the general solve algorithm may not be optimal because it does not take advantage of the fact that our matrix only has points along the diagonal or just on either side of it.  Such matrices commonly arise when solving differential equations and are called tridiagonal (or, more generally, band diagonal).  You can write your own tridiagonal solver or use SciPy's sparse matrix solvers to speed up the process, which are described at: https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.spsolve.html#scipy.sparse.linalg.spsolve

I have done the latter for you (because I am a nice guy). Look at the function `solve_tridiagonal()` in the homework template file.

Write a function `compute_spectrum()` that takes the following arguments:
* `z`: an $n$ element array positions within the atmosphere
* `rho`: an $n$ element array densities $\rho$ at these positions
* `temp`: an $n$ element array temperatures $T$ at these positions
* `nu`: an $m$ element array of frequencies on which the spectrum is evaluated
* `mu`: the cosine of the angle relative to the normal at which the spectrum is evaluated
* `taumin`: the value of the optical depth nearest the surface ($i=0$), with default value `0.001`
* `method`: sting corresponding to the solution method with options `npsolve` and `tridiag`. Specificying the former will have the function call `np.solve()` and the latter will call `solve_tridiagonal()`. The default value for `method` should be `npsolve`.

The function should set matrices `tau`, `bnu`, `enu`, etc. following the example above from problem 3.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

m = 200
nu = np.logspace(14., 17., m)
mu = 0.5

inu = hw.compute_spectrum(z, rho, temp, nu, mu)
plt.plot(nu, inu)
bnu = hw.planck(temp0, nu)
plt.plot(nu,bnu)
plt.xscale('log')
plt.yscale('log')
plt.ylim([1.e-6,1.]);

This form of spectrum is called a modifield blackbody spectrum. Note that it is below the blackbody at all frequencies and only approaches the blackbody at low fequencies where absorption opacity dominates.  Scattering makes an atmosphere a less efficient emitter.

What about performance. Does it matter which solver we use?

In [ ]:
# time with npsolve
%timeit inu = hw.compute_spectrum(z, rho, temp, nu, mu, taumin=0.001, method='npsolve')
# time with tridiagonal
%timeit inu = hw.compute_spectrum(z, rho, temp, nu, mu, taumin=0.001, method='tridiag')

For $n=200$, it doesn't really matter. What about if the matrix is bigger?

In [ ]:
n = 1000
z = np.linspace(zmin, zmax ,n)
l0 = (zmax - zmin) / np.log(taumax/taumin)
rho = taumin / (l0 * kapes) * np.exp((z-zmin)/l0)
temp = np.ones(n)*temp0


# time with npsolve
%timeit inu = hw.compute_spectrum(z, rho, temp, nu, mu, taumin=0.001, method='npsolve')

# time with tridiagonal
%timeit inu = hw.compute_spectrum(z, rho, temp, nu, mu, taumin=0.001, method='tridiag')

#This will take about a minute to complete with default %timeit settings

We can start to see a difference.  The solver matters when the size of the matrix begins to get very large.

## Problem 5

Write your own Gauss-Seidel implementation in a function name `my_gauss_seidel()`.  This function should *not* multiply ${\sf A}$ by its transpose. It should take arguments:
* `amat`: the $n \times n$ matrix ${\sf A}$
* `bvec`: the $n$ element vector $\mathbf{b}$
* `xinit`: and optional guess for the solution with default value `None`
* `eps`: stop iterating when the maximum relative changes is below this value. Default is `1.e-10`
* `itermax`: maximum number of iterations before stopping. Default is `10000`

This function should return the solution $\mathbf{x}$ to the equation ${\sf A} \mathbf{x} = \mathbf{b}$.

In [ ]:
# generate random matrix to check implementation
import numpy as np
n = 4
a = np.random.rand(n, n)
# make sure a is diagonally dominant (probably unnecessary)
for i in range(n):
    a[i,i] += 1
b = np.random.rand(n)
x = np.random.rand(n)
print("Matrix: ", a)
print("Vector: ", b)
x = hw.my_gauss_seidel(a, b)
print("Solution: ", x)
print("Error: ", np.dot(a, x)-b)

How does Gauss Seidel work for our transfer problem? 

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# compute matrices
# set atmosphere parameters
zmax = 1.e10
zmin = 0.
taumin = 1.e-3
taumax = 1.e3
kapes = 0.33
temp0 = 1.e5

# Create arrays
n = 100
z = np.linspace(zmin, zmax ,n)
l0 = (zmax - zmin) / np.log(taumax/taumin)
rho = taumin / (l0 * kapes) * np.exp((z-zmin)/l0)
temp = np.ones(n)*temp0

# evaluate and store functions of opacities, etc.
nu = 1.e15
abnu = hw.absorption_extinction(temp, rho, nu)
scnu = hw.scattering_extinction(rho)
bnu = hw.planck(temp, nu)
enu = abnu / (scnu + abnu)
    
# compute optical depth
dz = np.gradient(z)
tau = np.empty(n)
tau[0] = taumin
for i in range(1,n):
    tau[i] = tau[i-1] + (abnu[i-1]+scnu[i-1]+abnu[i]+scnu[i])*dz[i]

eps = np.ones(n)*0.5
amat, bvec = hw.transfer_matrices(tau, bnu, eps)

jnu4 = hw.my_gauss_seidel(amat, bvec, itermax=10000)
jnu5 = hw.my_gauss_seidel(amat, bvec, itermax=100000)
jnu_an = 1-np.exp(-np.sqrt(3*eps)*tau)/(1+np.sqrt(eps))
plt.plot(tau, jnu_an, 'r-')
plt.plot(tau, jnu4/bnu, 'b-.')
plt.plot(tau, jnu5/bnu, 'g:')
plt.xscale('log');

We see that convergence fails for $10^4$ iterations and we need to go even higher but it does converge to the solution before $10^5$. Hence, convergence is very slow on this problem, making it not very useful. For a problem like this, one would benefit from implementing a successive over-relaxation variant of Gauss-Seidel.

## Problem 6 (Grads Only)

Write a new function `lu_solve()` that takes a vector $\mathbf{b}$ and uses our in-class implementation of LU decomposition of $\sf A$ to solve the equation ${\sf A} \mathbf{x} = \mathbf{b}$. Note that I have already provided `lu_factorization()` for you.

In [ ]:
# generate random matrix to check implementation
import numpy as np
n = 4
a = np.random.rand(n, n)
# make sure a is diagonally dominant (probably unnecessary)
for i in range(n):
    a[i,i] += 1
b = np.random.rand(n)
x = np.random.rand(n)
print("Matrix: ", a)
print("Vector: ", b)
p, l, u = hw.lu_factorization(a)
x = hw.lu_solve(p, l, u, b)
print("Solution: ", x)
print("Error: ", np.dot(a, x)-b)

Does it work for our transfer problem?  Let's give it a try.  Add a new option for `method` called `lufact` to use the functions `lu_factorization()` and `lu_solve()` to compute `jnu`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

m = 200
nu = np.logspace(14., 17., m)
mu = 0.5

inu = hw.compute_spectrum(z, rho, temp, nu, mu, method='lufact')
plt.plot(nu, inu)
bnu = hw.planck(temp0, nu)
plt.plot(nu,bnu)
plt.xscale('log')
plt.yscale('log')
plt.ylim([1.e-6,1.]);

Your result should look like the answer above.  How fast is it?

In [ ]:
# using npsolve
%timeit inu = hw.compute_spectrum(z, rho, temp, nu, mu, taumin=0.001, method='npsolve')

# using tridiagonal
%timeit inu = hw.compute_spectrum(z, rho, temp, nu, mu, taumin=0.001, method='tridiag')

# using LU decomposition
%timeit inu = hw.compute_spectrum(z, rho, temp, nu, mu, taumin=0.001, method='lufact')

So, it is quite a bit slower.  No big surpise there. This would be true even we had removed the for loops and used numpy array options more efficiently. LU factorization is best when you are computing many different $\mathbf{b}$ vectors with the same $\sf A$ matrix.